<a href="https://colab.research.google.com/github/oliverlrj/NanoGPT-Math/blob/main/dpo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Step 1: Install necesscary packages

In [2]:
# If you have your own fork, use that URL instead:
!git clone https://github.com/oliverlrj/NanoGPT-Math.git || true
%cd NanoGPT-Math

# ---- Put the pretrained files into sft/ ----
from google.colab import files, output, pathlib
import os
os.makedirs("sft", exist_ok=True)
print("Upload sft/gpt.pt then sft/meta.pkl when prompted (two files).")
uploaded = files.upload()  # choose gpt.pt first or both at once
for name in uploaded:
    os.replace(name, f"sft/{name}")

# Quick check:
import os
assert os.path.exists("sft/gpt.pt"), "Missing sft/gpt.pt"
assert os.path.exists("sft/meta.pkl"), "Missing sft/meta.pkl"


Cloning into 'NanoGPT-Math'...
remote: Enumerating objects: 59, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 59 (delta 12), reused 9 (delta 8), pack-reused 40 (from 1)
Receiving objects: 100% (59/59), 1.69 MiB | 13.81 MiB/s, done.
Resolving deltas: 100% (22/22), done.
/content/NanoGPT-Math/NanoGPT-Math
Upload sft/gpt.pt then sft/meta.pkl when prompted (two files).


Saving gpt.pt to gpt.pt
Saving meta.pkl to meta.pkl


In [3]:
pip install matplotlib torch numpy transformers datasets tiktoken wandb tqdm

### Step 2: Package imports and configuration

In [4]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No CUDA")

True
1
Tesla T4


In [5]:
import sys
import os
sys.path.append(os.path.abspath(".."))
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import pickle
from model import GPT, GPTConfig
import random
from tqdm import tqdm
import time
import json
import matplotlib.pyplot as plt
# Configuration
beta = 0.5
device = 'cuda' if torch.cuda.is_available() else 'cpu'
base_lr = 1e-4
epochs = 5
batch_size = 64
max_length =64
num_samples = 1
max_new_tokens = 200
temperature = 0.8
top_k = 200
# tokenizer
with open("../sft/meta.pkl", "rb") as f:
    meta = pickle.load(f)
stoi, itos = meta["stoi"], meta["itos"]
def encode(s): return [stoi[c] for c in s]
def decode(l): return ''.join([itos[i] for i in l])

### Step 3: Define helper functions

In [6]:
def compute_logprob(input_ids):
    inputs = input_ids[:, :-1]
    targets = input_ids[:, 1:]
    logits, _ = gpt(inputs, full_seq=True)
    B, T, V = logits.size()
    logits_flat = logits.reshape(-1, V)
    targets_flat = targets.reshape(-1)
    loss = F.cross_entropy(logits_flat, targets_flat, ignore_index=0, reduction='none')
    loss = loss.reshape(B, T)
    attention_mask = (targets != 0).float()
    loss = (loss * attention_mask).sum(dim=1) / attention_mask.sum(dim=1)
    return -loss

def pad_or_truncate(seq, max_length):
    return seq[-max_length:] if len(seq) > max_length else seq + [0] * (max_length - len(seq))

def get_batches(lines, batch_size):
    random.shuffle(lines)
    #for l in lines:
    #    print(l[1])
    for i in range(0, len(lines), batch_size):
        batch = lines[i:i+batch_size]
        if len(batch) < batch_size:
            continue
        neg_inputs = [pad_or_truncate(encode(p['negative'] + '\n\n\n\n'), max_length) for p in batch]
        pos_inputs = [pad_or_truncate(encode(p['positive'] + '\n\n\n\n'), max_length) for p in batch]
        neg_tensor = torch.tensor(neg_inputs, dtype=torch.long, device=device)
        pos_tensor = torch.tensor(pos_inputs, dtype=torch.long, device=device)
        yield neg_tensor, pos_tensor

### Step 4: Load the pretrained NanoGPT model

In [8]:
ckpt = torch.load("./sft/gpt.pt", map_location=device)
gptconf = GPTConfig(**ckpt['model_args'])
gpt = GPT(gptconf)
state_dict = ckpt['model']
unwanted_prefix = '_orig_mod.'
for k in list(state_dict.keys()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
gpt.to(device).train()


GPT(
  (transformer): ModuleDict(
    (wte): Embedding(74, 348)
    (wpe): Embedding(256, 348)
    (drop): Dropout(p=0.2, inplace=False)
    (h): ModuleList(
      (0-5): 6 x Block(
        (ln_1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=348, out_features=1044, bias=False)
          (c_proj): Linear(in_features=348, out_features=348, bias=False)
          (attn_dropout): Dropout(p=0.2, inplace=False)
          (resid_dropout): Dropout(p=0.2, inplace=False)
        )
        (ln_2): LayerNorm()
        (mlp): MLP(
          (c_fc): Linear(in_features=348, out_features=1392, bias=False)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=1392, out_features=348, bias=False)
          (dropout): Dropout(p=0.2, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm()
  )
  (lm_head): Linear(in_features=348, out_features=74, bias=False)
)

### Step 5: Load Data (**students are required to complete this part!**)

The data is generated with `Generate_Data.py`, which creates random simple arithmetic and algebraic problems with the operators (+, -, /, *).

Positive responses are generated with deterministic functions and negative responses are generated with the pretrained SFT model.

100,000 pairs of positive-negative responses are generated.

In [12]:
# Load data from ../data/pos_neg_pairs.json
with open("data/pos_neg_pairs.json", "r") as f:
    lines = json.load(f)
print(f"Loaded {len(lines)} training pairs")

Loaded 100000 training pairs


### Step 6: Build the optimizer and scheduler (**students are required to complete this part!**)

We opted to use AdamW for our optimiser and StepLR for our scheduler.

AdamW adjusts the model's parameters (weights and biases) during training to minimize the loss function.

AdamW is preferred as it separates weight decay from the gradient updates, resulting in more effective regularization and better generalisation (less overfitting).

StepLR reduces the learning rate every epoch (LR gets multipled by `gamma` = 0.99 every epoch).

In [10]:
# Build optimizer and scheduler using AdamW
optimizer = torch.optim.AdamW(gpt.parameters(), lr=base_lr)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.99)
print(f"Optimizer: AdamW with lr={base_lr}")
print(f"Scheduler: StepLR with gamma=0.99")

Optimizer: AdamW with lr=0.0001
Scheduler: StepLR with gamma=0.99


### Step 7: Begin training (**students are required to complete this part!**)

The model's weights are adjusted based on the preference data.

The code addition was to convert the mathematical dpo loss formula into actual neural weight adjustments. It calculates a single loss value based on the margin between the preferred ($\log \pi(y_w)$) and rejected ($\log \pi(y_l)$) response log-probabilities.

It then pulls the weight in the positive direction to increase the probability of preferred answers and decreases probability of rejected answers.

In [11]:
total_steps = len(lines) // batch_size
for epoch in range(epochs):
    pbar = tqdm(get_batches(lines, batch_size))
    for step, (neg_tensor,pos_tensor) in enumerate(pbar):
        ###########################################################
        # Please complete the training code here!
        # Examples:
        # ...
        # neg_logprob
        # pos_logprob
        # loss = -F.logsigmoid((pos_logprob - neg_logprob) / beta).mean() - pos_logprob.mean() * 0.1
        # ...
        ###########################################################

        # move batch to device
        neg_tensor = neg_tensor.to(device)
        pos_tensor = pos_tensor.to(device)

        optimizer.zero_grad(set_to_none=True)

        # log-probs (helpers provided in template)
        neg_logprob = compute_logprob(neg_tensor)   # [B]
        pos_logprob = compute_logprob(pos_tensor)   # [B]
        loss = -F.logsigmoid((pos_logprob - neg_logprob) / beta).mean() - pos_logprob.mean() * 0.1

        loss.backward()
        torch.nn.utils.clip_grad_norm_(gpt.parameters(), 1.0)
        optimizer.step()

        # progress
        pbar.set_postfix(loss=f"{loss.item():.4f}",
                         lr=optimizer.param_groups[0]['lr'])


    scheduler.step()

    ckpt_path = f"./dpo/dpo.pt"
    torch.save({
        "model_state_dict": gpt.state_dict(),
        "model_args": ckpt['model_args'],
    }, ckpt_path)

1562it [04:03,  6.43it/s, loss=0.0226, lr=0.0001]
1562it [04:07,  6.30it/s, loss=0.0194, lr=9.9e-5]
1562it [04:07,  6.31it/s, loss=0.0163, lr=9.8e-5]
1562it [04:07,  6.30it/s, loss=0.0149, lr=9.7e-5]
1562it [04:06,  6.33it/s, loss=0.0147, lr=9.61e-5]


### Step 8: Begin testing (**students are required to complete this part!**)

The fine-tuned model is loaded and set to eval mode.

The test questions are encoded and fed into the mode with `gpt.generate`, and the response are decoded and the answers extracted.

In [22]:
# Load the fine-tuned model
ckpt_path = "./dpo/dpo.pt"
checkpoint = torch.load(ckpt_path, map_location=device)
gptconf = GPTConfig(**checkpoint['model_args'])
gpt = GPT(gptconf).to(device)
try:
    state_dict = checkpoint['model']
except:
    state_dict = checkpoint['model_state_dict']
unwanted_prefix = '_orig_mod.'
for k,v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
# Test
gpt.eval()
test_set = ["17+19=?", "3*17=?", "72/4=?", "72-x=34,x=?", "x*11=44,x=?", "3*17=?", "72/4=?", "72-x=34,x=?"]
with torch.no_grad():
    for prompt in test_set:
        prompt_ids = encode(prompt)
        ###########################################################
        # Please complete the test code here!
        # ...
        # gpt.generate(x, max_new_tokens, temperature=temperature, top_k=top_k)
        # ...
        ###########################################################
        x = torch.tensor([prompt_ids], dtype=torch.long, device=device)
        y = gpt.generate(x, max_new_tokens=50, temperature=temperature, top_k=top_k)

        # Decode the full output - flatten to 1D list
        out_ids = y[0].view(-1).tolist()
        out = decode(out_ids)

        # Extract just the answer (everything after the prompt)
        answer = out[len(prompt):].strip()
        # Take only the first line (stop at newline)
        answer = answer.split('\n')[0].strip()

        print(f"Q: {prompt}")
        print(f"A: {answer}")
        print()


Q: 17+19=?
A: The answer is 36 because 17+19 equals 36.

Q: 3*17=?
A: The answer is 51 because 3*17 equals 51.

Q: 72/4=?
A: The answer is 9 because 72/4 equals 9.

Q: 72-x=34,x=?
A: The answer is 48 because 72 minus 34 equals 4.

Q: x*11=44,x=?
A: The answer is 13 because 114 minus 11 equals 13.

Q: 3*17=?
A: The answer is 51 because 3*17 equals 51.

Q: 72/4=?
A: The answer is 9 because 72/4 equals 9.

Q: 72-x=34,x=?
A: The answer is 48 because 72 minus 34 equals 4.

